# Score new points from the support vectors

A trained kernel SVM keeps no weight vector. To score a new point it computes
$\sum_i \alpha_i y_i\, k(\mathbf{x}_i, \mathbf{x}) + b$, a sum over the support
vectors. Here you rebuild that score from what scikit-learn's `SVC` stores, and then
recover the bias $b$ yourself from the training points, without reading it from the
model.

**How to work through it.** Each part asks for one function. Replace the `TODO` and the
`raise NotImplementedError` with your code, then run the check cell under it. The check
prints `Part N passed` when your function is right, and stops with a message that says
what is wrong when it is not.

## Setup

Run this cell first.

In [ ]:
import numpy as np
from sklearn.datasets import make_moons
from sklearn.svm import SVC

X, y01 = make_moons(n_samples=80, noise=0.3, random_state=3)
y = np.where(y01 == 1, 1, -1)              # labels -1 and +1
gamma, C = 1.5, 1.0
svc = SVC(kernel="rbf", gamma=gamma, C=C).fit(X, y)
print(len(svc.support_), "support vectors out of", len(X))
T = np.random.default_rng(0).uniform(-1.5, 2.5, size=(40, 2))   # new points to score


def rbf_kernel(A, B, gamma):
    # exp(-gamma * squared distance) for every row of A against every row of B
    diff = np.asarray(A, dtype=float)[:, None, :] - np.asarray(B, dtype=float)[None, :, :]
    return np.exp(-gamma * (diff ** 2).sum(axis=2))

## Part 1 (coding task): Scores from the stored model

Write `score(svc, X_new, gamma)`, the score of every row of `X_new` as an array with one
entry per row, computed from `svc.support_vectors_`, `svc.dual_coef_` and
`svc.intercept_` with `rbf_kernel`. Do not call `svc.decision_function`: the check
compares against it.

In [ ]:
def score(svc, X_new, gamma):
    # TODO
    raise NotImplementedError

In [ ]:
expected = svc.decision_function(T)
_saved_df = svc.decision_function
def _blocked(*args, **kwargs):
    raise AssertionError("Part 1 forbids svc.decision_function: rebuild the score from support_vectors_, dual_coef_ and intercept_")
svc.decision_function = _blocked
try:
    s = score(svc, T, gamma)
finally:
    svc.decision_function = _saved_df
assert np.shape(s) == (len(T),), f"expected one score per new point, shape ({len(T)},), got {np.shape(s)}"
assert not np.allclose(s + svc.intercept_[0], expected), "the bias is missing: add svc.intercept_[0] to every score"
labels_again = rbf_kernel(T, svc.support_vectors_, gamma) @ (svc.dual_coef_[0] * y[svc.support_]) + svc.intercept_[0]
assert not np.allclose(s, labels_again), "these scores drop the sign of each label: dual_coef_ already holds alpha_i times y_i, so multiplying by y_i again (or taking absolute values) gives alpha_i alone"
assert np.allclose(s, expected, atol=1e-6), f"scores differ from svc.decision_function by up to {np.abs(s - expected).max():.3f}"
print("Part 1 passed")


## Part 2 (coding task): The bias from the training points

`SVC` reports $b$ in `svc.intercept_`. Recover it yourself, from the training points
`X`, `y`, the multipliers stored in `svc` and the kernel. Write `intercept(svc, X, y, C, gamma)`.

A single training point can fix $b$ when you know its score, but the solver stops at a
small tolerance, so the value it gives differs slightly from point to point. Average the
values over every training point that can be used. Do not read `svc.intercept_`.

In [ ]:
def intercept(svc, X, y, C, gamma):
    # TODO
    raise NotImplementedError

In [ ]:
_b_ref = float(svc.intercept_[0])
_df_ref = svc.decision_function(T)
_saved_ic = svc.intercept_
del svc.intercept_
try:
    b = intercept(svc, X, y, C, gamma)
finally:
    svc.intercept_ = _saved_ic
assert np.ndim(b) == 0, f"b should be one number, got shape {np.shape(b)}"
gap = abs(b - _b_ref)
assert gap < 1e-4, f"b = {b:.4f} is off from the solver's {_b_ref:.4f} by {gap:.4f}: not every support vector has a score of exactly +1 or -1"
s_own = rbf_kernel(T, svc.support_vectors_, gamma) @ svc.dual_coef_[0] + b
assert np.allclose(s_own, _df_ref, atol=1e-4), "with your b the scores do not match decision_function"
alpha = np.abs(svc.dual_coef_[0])
print(f"b = {b:.6f}, from {np.sum(alpha < C * (1 - 1e-8))} of the {len(alpha)} support vectors")
print("Part 2 passed")


## Solution

### Part 1: Scores from the stored model

In [ ]:
def score(svc, X_new, gamma):
    K = rbf_kernel(X_new, svc.support_vectors_, gamma)   # one column per support vector
    return K @ svc.dual_coef_[0] + svc.intercept_[0]      # dual_coef_ is alpha_i * y_i

**Why it works.** `rbf_kernel(X_new, svc.support_vectors_, gamma)` has one row per new point and
one column per support vector, and `dual_coef_` already carries each $\alpha_i y_i$, so one
product sums the terms. The other 44 training points have $\alpha_i = 0$ and are not stored
at all.

### Part 2: The bias from the training points

In [ ]:
def intercept(svc, X, y, C, gamma):
    alpha_y = svc.dual_coef_[0]
    alpha = np.abs(alpha_y)
    on_margin = alpha < C * (1 - 1e-8)                   # 0 < alpha < C: score exactly +1 or -1
    idx = svc.support_[on_margin]
    without_b = rbf_kernel(X[idx], svc.support_vectors_, gamma) @ alpha_y
    return np.mean(y[idx] - without_b)

**Why it works.** A point with $0 < \alpha_i < C$ sits exactly on the edge of its margin, so its score
is $y_i$: $\sum_j \alpha_j y_j k(\mathbf{x}_j, \mathbf{x}_i) + b = y_i$, which gives
$b = y_i - \sum_j \alpha_j y_j k(\mathbf{x}_j, \mathbf{x}_i)$. Here 13 of the 36 support
vectors are of that kind, and their average matches `svc.intercept_` to about $10^{-9}$.
The other 23 have $\alpha_i = C$: they sit inside the margin or on the wrong side, their
scores are not $\pm1$, and including them moves $b$ from $-0.070$ to $-0.092$. The points
with $\alpha_i = 0$ are only known to clear the margin, so they cannot fix $b$ either.